In [1]:
!git clone https://github.com/WeilunWang/semantic-diffusion-model
%cd semantic-diffusion-model
!pip install -q flask pyngrok gdown Pillow numpy flask-ngrok
!pip install -q torch==2.4.0 torchvision --index-url https://download.pytorch.org/whl/cu121


fatal: destination path 'semantic-diffusion-model' already exists and is not an empty directory.
/content/semantic-diffusion-model


check if still exists
https://drive.google.com/file/d/1O8Avsvfc8rP9LIt5tkJxowMTpi1nYiik/view?usp=drive_link


In [2]:
!pip install -q gdown

import gdown
import os

os.makedirs("./net_sdm_ade20k/sdm_ade20k", exist_ok=True)

gdown.download(
    id="1O8Avsvfc8rP9LIt5tkJxowMTpi1nYiik",
    output = "./net_sdm_ade20k/sdm_ade20k/ema_0.9999_best.pt",
    quiet=False
)


Downloading...
From (original): https://drive.google.com/uc?id=1O8Avsvfc8rP9LIt5tkJxowMTpi1nYiik
From (redirected): https://drive.google.com/uc?id=1O8Avsvfc8rP9LIt5tkJxowMTpi1nYiik&confirm=t&uuid=8d0e53fd-8c24-4688-ae64-e75c41f7670c
To: /content/semantic-diffusion-model/net_sdm_ade20k/sdm_ade20k/ema_0.9999_best.pt
100%|██████████| 2.45G/2.45G [00:42<00:00, 57.0MB/s]


'./net_sdm_ade20k/sdm_ade20k/ema_0.9999_best.pt'

In [3]:

import zipfile
import sys
import os


import torch

import base64
import io
import shutil

file_path = "./net_sdm_ade20k/sdm_ade20k/ema_0.9999_best.pt"
extract_dir = "./extracted_checkpoint"

import struct


outer_zip = "./net_sdm_ade20k/sdm_ade20k/ema_0.9999_best.pt"
extract_dir = "./extracted_inner"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(outer_zip, 'r') as zf:
    zf.extractall(extract_dir)

inner_file = os.path.join(extract_dir, "sdm_ade20k", "ema_0.9999_best.pt")
target_file = "./net_sdm_ade20k/sdm_ade20k/ema_0.9999_best_WORKING.pt"

shutil.copy2(inner_file, target_file)



'./net_sdm_ade20k/sdm_ade20k/ema_0.9999_best_WORKING.pt'

In [4]:
import torch
import gc
import os
import subprocess

torch.cuda.empty_cache()
gc.collect()

!pkill -f ngrok
!pkill -f flask
!lsof -ti:5050 | xargs kill -9 2>/dev/null

import time
time.sleep(2)
print("cleanup complete")

cleanup complete


In [5]:
import torch
import torch.nn.functional as F
import base64
import io
import numpy as np
import traceback
from PIL import Image
from flask import Flask, request, jsonify
from pyngrok import ngrok
import sys

sys.path.insert(0, '/content/semantic-diffusion-model')

from guided_diffusion.script_util import (
    model_and_diffusion_defaults,
    create_model_and_diffusion,
    args_to_dict,
)
from types import SimpleNamespace

args = model_and_diffusion_defaults()
args.update({
    "attention_resolutions": "32,16,8",
    "class_cond": True,
    "diffusion_steps": 1000,
    "image_size": 256,
    "learn_sigma": True,
    "noise_schedule": "linear",
    "num_channels": 256,
    "num_head_channels": 64,
    "num_res_blocks": 2,
    "resblock_updown": True,
    "use_fp16": True,
    "use_scale_shift_norm": True,
    "timestep_respacing": "25",
    "dropout": 0.0,
    "use_checkpoint": True,
    "num_classes": 151,
    "dataset_mode": "ade20k",
    "no_instance": True,
})


args_ns = SimpleNamespace(**args)

model, diffusion = create_model_and_diffusion(
    **args_to_dict(args_ns, model_and_diffusion_defaults().keys())
)

checkpoint_path = "./net_sdm_ade20k/sdm_ade20k/ema_0.9999_best_WORKING.pt"
state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
model.load_state_dict(state_dict)

model = model.cuda()
model = model.half()
model.eval()

torch.cuda.empty_cache()
print("model loaded")


model loaded


In [6]:
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from types import SimpleNamespace
import sys

sys.path.insert(0, '/content/semantic-diffusion-model')

from guided_diffusion.script_util import (
    model_and_diffusion_defaults,
    create_model_and_diffusion,
    args_to_dict,
)

print("Loading model in FP32 mode...")

args = model_and_diffusion_defaults()
args.update({
    "attention_resolutions": "32,16,8",
    "class_cond": True,
    "diffusion_steps": 1000,
    "image_size": 256,
    "learn_sigma": True,
    "noise_schedule": "linear",
    "num_channels": 256,
    "num_head_channels": 64,
    "num_res_blocks": 2,
    "resblock_updown": True,
    "use_fp16": False,
    "use_scale_shift_norm": True,
    "timestep_respacing": "50",
    "dropout": 0.0,
    "use_checkpoint": True,
    "num_classes": 151,
    "dataset_mode": "ade20k",
    "no_instance": True,
})

args_ns = SimpleNamespace(**args)

model, diffusion = create_model_and_diffusion(
    **args_to_dict(args_ns, model_and_diffusion_defaults().keys())
)

checkpoint_path = "./net_sdm_ade20k/sdm_ade20k/ema_0.9999_best_WORKING.pt"
state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
model.load_state_dict(state_dict)

model = model.cuda()
#usig fp32
model = model.float()
model.eval()

print(f"model loaded")


Loading model in FP32 mode...
model loaded


In [7]:
import time
from datetime import datetime

app = Flask(__name__)

def encode_image(arr: np.ndarray) -> str:
    img = Image.fromarray(arr.astype(np.uint8))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy", "model_loaded": True})

import time

@app.route("/generate_batch", methods=["POST"])
def generate_batch():
    print("[/generate_batch] called")

    batch_start = time.time()

    try:
        data = request.get_json(force=True, silent=True)
        if not data or "frames" not in data:
            return jsonify({"error": "Missing 'frames' field"}), 400

        masks_b64 = data["frames"]
        print(f"frames received: {len(masks_b64)}")
        results = []
        timings = []

        for idx, mask_b64 in enumerate(masks_b64):
            frame_start = time.time()

            try:
                mask_bytes = base64.b64decode(mask_b64)
                mask_pil = Image.open(io.BytesIO(mask_bytes)).convert("L")
                mask_np = np.array(mask_pil)
                orig_h, orig_w = mask_np.shape[:2]
                decode_time = time.time() - t0

                mask_resized = np.array(Image.fromarray(mask_np).resize((256, 256), Image.NEAREST))
                mask_resized = np.clip(mask_resized, 0, 150)

                label_indices = torch.from_numpy(mask_resized).long().unsqueeze(0)
                label_one_hot = F.one_hot(label_indices, num_classes=151)
                label_one_hot = label_one_hot.permute(0, 3, 1, 2).float()
                label = label_one_hot.cuda()
                prep_time = time.time() - t0

                t0 = time.time()
                with torch.no_grad():
                    sample = diffusion.p_sample_loop(
                        model,
                        (1, 3, 256, 256),
                        clip_denoised=True,
                        model_kwargs={"y": label},
                        progress=False,
                    )
                gen_time = time.time() - t0

                t0 = time.time()
                out = ((sample + 1) / 2 * 255).clamp(0, 255)
                out_np = out.squeeze(0).permute(1, 2, 0).cpu().numpy().astype(np.uint8)
                out_np = np.array(Image.fromarray(out_np).resize((orig_w, orig_h), Image.BICUBIC))
                result_b64 = encode_image(out_np)
                post_time = time.time() - t0

                results.append(result_b64)

                timings.append({
                    "frame": idx,
                    "decode_ms": round(decode_time * 1000, 1),
                    "prep_ms": round(prep_time * 1000, 1),
                    "gen_s": round(gen_time, 2),
                    "post_ms": round(post_time * 1000, 1),
                })

                print(f"frame {idx}: gen={gen_time:.2f}s")

                del sample, label, label_one_hot, out, out_np
                torch.cuda.empty_cache()

            except Exception as e:
                print(f"frame {idx} failed: {e}")
                timings.append({"frame": idx, "error": str(e)})
                results.append(None)

        batch_total = time.time() - batch_start
        valid_count = len([r for r in results if r is not None])

        print(f"batch complete: {valid_count}/{len(masks_b64)} frames, time={batch_total:.2f}s")

        return jsonify({
            "frames": [r for r in results if r is not None],
            "stats": {
                "total": len(masks_b64),
                "success": valid_count,
                "batch_time_s": round(batch_total, 2),
                "avg_time_s": round(batch_total / valid_count, 2) if valid_count > 0 else 0,
                "timings": timings
            }
        })

    except Exception as e:
        print(f"batch error: {e}")
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

@app.route("/generate", methods=["POST"])
def generate():
    try:
        data = request.get_json(force=True, silent=True)
        if not data or "mask" not in data:
            return jsonify({"error": "Missing 'mask'"}), 400

        mask_bytes = base64.b64decode(data["mask"])
        mask_pil = Image.open(io.BytesIO(mask_bytes)).convert("L")
        mask_np = np.array(mask_pil)

        orig_h, orig_w = mask_np.shape[:2]

        mask_resized = np.array(Image.fromarray(mask_np).resize((256, 256), Image.NEAREST))
        mask_resized = np.clip(mask_resized, 0, 150)

        label_indices = torch.from_numpy(mask_resized).long().unsqueeze(0)
        label_one_hot = F.one_hot(label_indices, num_classes=151)
        label_one_hot = label_one_hot.permute(0, 3, 1, 2).float()
        label = label_one_hot.cuda()
        t0 = time.time()
        with torch.no_grad():
            sample = diffusion.p_sample_loop(
                model,
                (1, 3, 256, 256),
                clip_denoised=True,
                model_kwargs={"y": label},
                progress=False,
            )
        gen_time = time.time() - t0
        out = ((sample + 1) / 2 * 255).clamp(0, 255)
        out_np = out.squeeze(0).permute(1, 2, 0).cpu().numpy().astype(np.uint8)

        out_np = np.array(Image.fromarray(out_np).resize((orig_w, orig_h), Image.BICUBIC))

        del sample, label, label_one_hot
        torch.cuda.empty_cache()

        print(f"generated: {orig_w}x{orig_h}")
        return jsonify({"image": encode_image(out_np)})

    except Exception as e:
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500

ngrok.kill()
ngrok.set_auth_token("3CyVDZM1RIAIUDXAvrqbKtufwZH_4axPNYua3yJoV7goxJXZF")

public_url = ngrok.connect(5000).public_url
print(f"PUBLIC URL: {public_url}\n")

app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

PUBLIC URL: https://send-krypton-straining.ngrok-free.dev

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
